In [1]:
import pandas as pd
import numpy as np
import sys
import os
import nltk
sys.path.append('..')
from src.data_loader import NewsTokenizer, load_glove, parse_behaviors

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\costellodt\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\costellodt\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
news_cols = ['news_id', 'category', 'subcategory', 'title',
             'abstract', 'url', 'title_entities', 'abstract_entities']
news_df = pd.read_csv('../data/MINDsmall_train/news.tsv',
                       sep='\t', names=news_cols)

beh_cols = ['impression_id', 'user_id', 'time', 'history', 'impressions']
behaviors_df = pd.read_csv('../data/MINDsmall_train/behaviors.tsv',
                            sep='\t', names=beh_cols)

In [3]:
tokenizer = NewsTokenizer(max_title_len=30, min_word_freq=2)
tokenizer.build_vocab(news_df['title'].dropna())

news_encoded = {}
for _, row in news_df.iterrows():
    if pd.notna(row['title']):
        news_encoded[row['news_id']] = tokenizer.encode_title(row['title'])

Vocabulary size: 20774


In [4]:
behaviors_dev_df = pd.read_csv('../data/MINDsmall_dev/behaviors.tsv',
                                sep='\t', names=beh_cols)
news_dev_df = pd.read_csv('../data/MINDsmall_dev/news.tsv',
                           sep='\t', names=news_cols)

samples_dev = parse_behaviors(behaviors_dev_df, news_encoded, neg_k=4)
print(f'Total validation samples: {len(samples_dev)}')

Total validation samples: 111383


In [5]:
samples = parse_behaviors(behaviors_df, news_encoded, neg_k=4)
print(f'Total training samples: {len(samples)}')

Total training samples: 236344


In [6]:
embedding_matrix = load_glove(
    '../data/glove/glove.6B.300d.txt',
    tokenizer.word2idx,
    embed_dim=300
)

Found 19069/20774 words in GloVe


In [7]:
import pickle

os.makedirs('../data/processed', exist_ok=True)

with open('../data/processed/news_encoded.pkl', 'wb') as f:
    pickle.dump(news_encoded, f)

with open('../data/processed/samples.pkl', 'wb') as f:
    pickle.dump(samples, f)

with open('../data/processed/samples_dev.pkl', 'wb') as f:
    pickle.dump(samples_dev, f)

np.save('../data/processed/embedding_matrix.npy', embedding_matrix)

with open('../data/processed/word2idx.pkl', 'wb') as f:
    pickle.dump(tokenizer.word2idx, f)

print('All processed data saved!')

All processed data saved!
